# Problemas de Otimização: Restaurante com Delivery

Modelos resolvidos em Python com `scipy.optimize` (sem Excel Solver). Roda direto no Google Colab, sem instalar nada.

- **Problema 1:** Maximização de Lucro, com foco na **experiência do cliente**
- **Problema 2:** Minimização de Custos, com foco no **gargalo logístico**

Os dados reais vêm de `tabela_unificada_2026_01_bruto.csv` (46 pedidos/dia, 19,2% de cancelamento, R$ 44,00 de reembolso e perda por pedido cancelado). O restante são premissas do grupo.

## Problema 1: Maximização de Lucro

**Contexto:** o restaurante recebe 46 pedidos por dia no delivery e precisa decidir quantos deles recebem atendimento **padrão** e quantos recebem atendimento com **experiência garantida** (confirmação ativa do status do pedido, alerta de atraso e embalagem reforçada). A experiência garantida custa mais por pedido e consome tempo do operador, mas cancela menos. Cada cancelamento custa R$ 44,00 em reembolso e perda.

**Variáveis de decisão:** $x_P$ = pedidos/dia com atendimento padrão; $x_G$ = pedidos/dia com experiência garantida.

**Função Objetivo:** Maximizar $Z = 9{,}328 \cdot x_P + 13{,}72 \cdot x_G$

(lucro esperado por pedido = $(1-p)\cdot$margem $- p \cdot$ custo do cancelamento $-$ custo do serviço)

**Restrições:**
1. Demanda: $x_P + x_G \le 46$
2. Horas do operador (4 min por pedido garantido): $4\,x_G \le 120$
3. Nível de serviço (cancelamento esperado $\le$ 15%): $0{,}042\,x_P - 0{,}07\,x_G \le 0$
4. Não-negatividade e inteiros: $x_P, x_G \ge 0$

In [ ]:
import numpy as np
from scipy.optimize import milp, LinearConstraint, Bounds

# ---------- Dados ----------
TICKET = 55.00                 # ticket padrão (R$)
MARGEM = 0.40 * TICKET         # premissa: margem de contribuição de 40% = R$ 22,00
P_PADRAO = 0.192               # dado real: 19,2% de cancelamento no atendimento padrão
P_GARANTIDO = 0.08             # premissa: 8% de cancelamento com experiência garantida
CUSTO_CANCEL = 44.00           # dado real: reembolsos e perdas / pedidos cancelados
CUSTO_SERVICO = 3.00           # premissa: R$ extras por pedido garantido

lucro_padrao = (1 - P_PADRAO) * MARGEM - P_PADRAO * CUSTO_CANCEL                       # R$ 9,328
lucro_garantido = (1 - P_GARANTIDO) * MARGEM - P_GARANTIDO * CUSTO_CANCEL - CUSTO_SERVICO   # R$ 13,72

# ---------- Modelo ----------
# Variáveis de decisão (inteiras, pois não se entrega meio pedido): x = [xP, xG]
# Função Objetivo: o milp minimiza, então maximizar o lucro equivale a minimizar o lucro negativo
c = [-lucro_padrao, -lucro_garantido]

# Restrições (A @ x <= b)
A = [[1, 1],                                   # 1. Demanda
     [0, 4],                                   # 2. Horas do operador (minutos)
     [P_PADRAO - 0.15, P_GARANTIDO - 0.15]]    # 3. Nível de serviço
b = [46, 120, 0]

# ---------- Resolução ----------
res = milp(c, constraints=LinearConstraint(A, -np.inf, b),
           integrality=[1, 1], bounds=Bounds(0, np.inf))   # 3. Não-negatividade

xP, xG = np.round(res.x).astype(int)
print(f"Status: {res.message}")
print(f"Qtd Atendimento Padrão: {xP}")
print(f"Qtd Experiência Garantida: {xG}")
print(f"Lucro Máximo: R$ {-res.fun:.2f} por dia")
print(f"Lucro de hoje (tudo no padrão): R$ {lucro_padrao * 46:.2f} | ganho: R$ {-res.fun - lucro_padrao * 46:.2f} por dia")

### Análise de Cenário (What-If)
**Cenário:** e se o operador de acompanhamento faltar metade do turno e as horas caírem de 120 para 60 minutos por dia?

In [ ]:
# What-If: alterando a restrição 2 de <= 120 para <= 60
b_whatif = [46, 60, 0]

res_w = milp(c, constraints=LinearConstraint(A, -np.inf, b_whatif),
             integrality=[1, 1], bounds=Bounds(0, np.inf))

xP_w, xG_w = np.round(res_w.x).astype(int)
print(f"Status: {res_w.message}")
print(f"Qtd Atendimento Padrão: {xP_w}")
print(f"Qtd Experiência Garantida: {xG_w}")
print(f"Pedidos atendidos: {xP_w + xG_w} de 46")
print(f"Lucro Máximo: R$ {-res_w.fun:.2f} por dia")
print(f"Queda de lucro: R$ {-res.fun - (-res_w.fun):.2f} por dia")

**Resultado:** com 60 minutos de operador, apenas 15 pedidos podem ser acompanhados (4 min cada). Para manter o cancelamento esperado abaixo de 15%, o algoritmo passa a **atender só 40 dos 46 pedidos** (25 padrão e 15 garantidos), e o lucro cai de **R$ 560,85 para R$ 439,00 por dia** (queda de R$ 121,85). Sem o acompanhamento do operador, a restrição de nível de serviço deixa de ser cumprida.

## Problema 2: Minimização de Custos

**Contexto:** os 46 pedidos diários precisam sair de fontes de entrega para 3 zonas: Norte (16 pedidos), Centro (18) e Sul (12). Hoje só a **Cozinha Central** despacha, com 2 motoboys (28 pedidos/dia). O que sobra vai para um **motoboy de reforço**, que é caro (R$ 14,00 por pedido). A proposta é abrir um **Ponto de Apoio** perto da Zona Sul, com 1 motoboy (14 pedidos/dia) e custo fixo de R$ 60,00/dia, e deixar o algoritmo decidir se compensa.

**Variáveis de decisão:** $x_{ij}$ = pedidos despachados da fonte $i$ (Central, Apoio, Reforço) para a zona $j$ (Norte, Centro, Sul); $y \in \{0,1\}$ = abrir o Ponto de Apoio.

**Função Objetivo:** Minimizar

$C = 4{,}8x_{C,N} + 8{,}0x_{C,C} + 12x_{C,S} + 11{,}2x_{A,N} + 7{,}2x_{A,C} + 4{,}0x_{A,S} + 14\,(x_{R,N}+x_{R,C}+x_{R,S}) + 60\,y$

**Restrições:**
1. Demanda: $\sum_i x_{iN}=16$, $\sum_i x_{iC}=18$, $\sum_i x_{iS}=12$
2. Capacidade da Central: $\sum_j x_{Central,j} \le 28$
3. Capacidade do Apoio: $\sum_j x_{Apoio,j} \le 14\,y$
4. Não-negatividade e inteiros: $x_{ij} \ge 0$

In [ ]:
# ---------- Dados ----------
fontes = ["Central", "Apoio", "Reforco"]
zonas = ["Norte", "Centro", "Sul"]

# Custo por pedido (R$): 0,40 por minuto de rota; reforço custa R$ 14,00 em qualquer zona
custos = [[4.8, 8.0, 12.0],     # Central -> Norte, Centro, Sul
          [11.2, 7.2, 4.0],     # Apoio   -> Norte, Centro, Sul
          [14.0, 14.0, 14.0]]   # Reforço -> Norte, Centro, Sul
demanda = [16, 18, 12]          # pedidos/dia por zona (total de 46)
cap_central = 28                # 2 motoboys x 14 pedidos/dia
cap_apoio = 14                  # 1 motoboy x 14 pedidos/dia
custo_fixo_apoio = 60.00        # R$/dia

def resolver(cap_apoio, forcar_apoio_aberto=None):
    """Vetor de decisão: 9 variáveis x_ij (linha por fonte) + 1 variável y (abrir o Apoio)."""
    c = np.r_[np.array(custos).flatten(), custo_fixo_apoio]           # Função Objetivo
    A, lo, hi = [], [], []
    for j in range(3):                                                 # 1. Demanda (igualdade)
        linha = np.zeros(10); linha[[j, 3 + j, 6 + j]] = 1
        A.append(linha); lo.append(demanda[j]); hi.append(demanda[j])
    linha = np.zeros(10); linha[0:3] = 1                               # 2. Capacidade da Central
    A.append(linha); lo.append(-np.inf); hi.append(cap_central)
    linha = np.zeros(10); linha[3:6] = 1; linha[9] = -cap_apoio        # 3. Capacidade do Apoio (<= cap * y)
    A.append(linha); lo.append(-np.inf); hi.append(0)
    lb = np.zeros(10); ub = np.r_[np.full(9, np.inf), 1]
    if forcar_apoio_aberto is not None:
        lb[9] = ub[9] = forcar_apoio_aberto
    r = milp(c, constraints=LinearConstraint(np.array(A), lo, hi),
             integrality=np.ones(10), bounds=Bounds(lb, ub))           # 4. Não-negatividade
    return np.round(r.x[:9]).astype(int).reshape(3, 3), r.fun, int(round(r.x[9])), r.message

# ---------- Hoje: sem Ponto de Apoio ----------
x_hoje, custo_hoje, _, _ = resolver(cap_apoio, forcar_apoio_aberto=0)

# ---------- Resolução: o algoritmo decide ----------
x, custo, abrir, status = resolver(cap_apoio)
print(f"Status: {status}")
print(f"Abrir o Ponto de Apoio? {'Sim' if abrir else 'Não'}")
for i, fonte in enumerate(fontes):
    print(f"  {fonte:<8}", {zona: int(x[i][j]) for j, zona in enumerate(zonas)})
print(f"Pedidos por reforço: {x[2].sum()} (hoje: {x_hoje[2].sum()})")
print(f"Custo Mínimo: R$ {custo:.2f} por dia (hoje: R$ {custo_hoje:.2f}) | economia: R$ {custo_hoje - custo:.2f} por dia")

### Análise de Cenário (What-If)
**Cenário:** e se o Ponto de Apoio estiver aberto (custo fixo já pago), mas o motoboy do dia não comparecer? Nos dados de 2026, isso aconteceu em 23 dias de 181 (cerca de 13% dos dias).

In [ ]:
# What-If: a capacidade do Apoio cai de 14 para 0, com o ponto aberto (y = 1, custo fixo pago)
x_w, custo_w, abrir_w, status_w = resolver(cap_apoio=0, forcar_apoio_aberto=1)

print(f"Status: {status_w}")
for i, fonte in enumerate(fontes):
    print(f"  {fonte:<8}", {zona: int(x_w[i][j]) for j, zona in enumerate(zonas)})
print(f"Pedidos por reforço: {x_w[2].sum()}")
print(f"Custo Mínimo: R$ {custo_w:.2f} por dia")
print(f"Aumento de custo vs. plano normal: R$ {custo_w - custo:.2f} por dia")

**Resultado:** sem o motoboy do Apoio, os pedidos voltam ao reforço (**18 pedidos**, em vez de 4) e o custo sobe de **R$ 351,20 para R$ 484,80 por dia** (+R$ 133,60), valor até maior que os R$ 424,80 de hoje, porque o custo fixo do Apoio continua sendo pago. A lição para o plano: ter **um motoboy reserva** para o Ponto de Apoio.